# AutiLens — Phase 3: LoRA fine-tuning on A100

Phase 1 (frozen backbone + cached features) runs locally in minutes and does **not**
belong here. This notebook is only for LoRA, where the backbone weights change and
features can no longer be cached.

**Before running:** upload the packed cache to your **own private Google Drive**.
Build it locally with:
```bash
python -m scripts.pack_for_colab --out data/packed          # ~430 MB
python -m scripts.pack_for_colab --validate --out data/packed   # GATE — must PASS
```

> **Privacy.** This is video of real children sourced from YouTube/Facebook.
> Private Drive only — never GitHub, never a public dataset host (guide §14).

**Methodology carries over unchanged:** nested CV, thresholds/calibrators fitted on
the inner split and frozen before the outer fold is scored, and the 92-clip test set
is never touched here — selection is on outer OOF only.


## 1. Check the GPU


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
import torch; print('torch', torch.__version__, '| cuda', torch.cuda.is_available())


## 2. Clone the repo (code only — no data in git)


In [ ]:
import os
REPO = 'https://github.com/YOUR_USERNAME/AutiLens.git'   # <-- set this
if not os.path.exists('AutiLens'):
    !git clone $REPO AutiLens
%cd AutiLens
!pip -q install librosa opencv-python-headless pyyaml tqdm scikit-learn


## 3. Mount Drive and unpack

Expects `windows_q95.tar` and `mel_f16.npz` in the Drive folder below.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PACK = '/content/drive/MyDrive/autilens_packed'   # <-- set this
!mkdir -p data/processed/windows_jpg
!tar -xf $PACK/windows_q95.tar -C data/processed/windows_jpg
!cp $PACK/mel_f16.npz data/processed/
!du -sh data/processed/windows_jpg


## 4. Re-run the packed-cache gate on this machine

The gate was run locally, but decode paths differ between machines. If any
thresholded decision flips here, stop and use the raw cache.


In [ ]:
!python -m scripts.pack_for_colab --validate --out $PACK --n 40 || print('GATE FAILED — use the raw cache')


## 5. Confirm the LoRA adapter is wired correctly

torchvision's Swin3D reads `qkv.weight` directly and passes it to a functional, so an
adapter overriding `forward()` silently does nothing. The audit below is what catches
that: base weights frozen, only A/B trainable.


In [ ]:
!python -m src.cv_lora --stages last --rank 8
!python -m src.cv_lora --stages all  --rank 8


## 6. Train

Start with variant **A** (`--stages last`, ~74K trainable). Expand only if it helps —
335 training clips is very little for a 28M-param backbone.

| variant | stages | rank | trainable | ~cost on A100 |
|---|---|---|---|---|
| A | last | 8 | 74 K | ~10 min |
| B | all | 8 | 212 K | ~20 min |
| C | all | 16 | 424 K | ~25 min |


In [ ]:
# Variant A — no --eval-test: selection is on outer OOF only.
!python -m src.cv_lora --stages last --rank 8 --target families --tag lora_A


In [ ]:
# Variants B and C, only if A shows promise.
# !python -m src.cv_lora --stages all --rank 8  --target families --tag lora_B
# !python -m src.cv_lora --stages all --rank 16 --target families --tag lora_C


## 7. Compare — on OOF, never on test


In [ ]:
!python -m src.evaluation.summary


## 8. Copy results back to Drive

Checkpoints and reports only; no source video leaves this session.


In [ ]:
!mkdir -p $PACK/results
!cp -v models/lora_*.pt models/lora_*_oof.npz reports/lora_*_cv.json $PACK/results/ 2>/dev/null
!ls -la $PACK/results


## 9. Watch for overfitting

If inner-val macro-F1 runs far ahead of outer OOF, the adapter is memorising 335
clips — drop the rank or the epoch count. A LoRA variant that fails to beat the
frozen model is a valid, reportable result: Phase 4 gates on measurement, so nothing
weak ships.
